# Session 1: How AI Models Work - Companion Notebook

**Hands-on code demos** for Session 1 of the AI Engineering Teaching Series.

See `session_1_how_ai_models_work.md` for detailed theory.

## Learning Objectives

By the end of this notebook, you will:
- Tokenize text and estimate costs with tiktoken
- Make generation and embedding API calls
- Measure cosine similarity and build a mini semantic search
- See streaming inference in action
- Compare temperature settings visually

## Table of Contents

1. [Setup](#setup)
2. [Tokenization with tiktoken](#tokenization)
3. [LLM API Calls: Generation and Embedding](#api-calls)
4. [Cosine Similarity and Semantic Search](#cosine-similarity)
5. [Streaming Inference](#streaming)
6. [Temperature Comparison](#temperature)
7. [Attention Heatmap Visualization](#attention)
8. [Exercises](#exercises)

---
## 1. Setup <a id='setup'></a>

In [ ]:
# Setup: install dependencies and configure API key
import os
import sys
from pathlib import Path

# Add parent directories to path for imports
sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent.parent / ".env")

print("Setup complete!")
print(f"OpenAI key: {'found' if os.getenv('OPENAI_API_KEY') else 'NOT FOUND - add to .env'}")

---
## 2. Tokenization with tiktoken <a id='tokenization'></a>

tiktoken is OpenAI's tokenizer library. It lets you see exactly how text is split into tokens,
count tokens before making API calls, and estimate costs.

In [ ]:
# Tokenize text and inspect the results
import tiktoken

# gpt-4o and gpt-4o-mini use the "o200k_base" encoding
enc = tiktoken.encoding_for_model("gpt-4o-mini")

text = "Hello, world! This is a test of tokenization."
tokens = enc.encode(text)

print(f"Text: {text}")
print(f"Token count: {len(tokens)}")
print(f"Token IDs: {tokens}")
print()

# Decode each token to see the subword units
for token_id in tokens:
    print(f"  {token_id:>6} -> '{enc.decode([token_id])}'")

In [ ]:
# Compare token counts across languages
texts = {
    "English": "The quick brown fox jumps over the lazy dog.",
    "Spanish": "El rapido zorro marron salta sobre el perro perezoso.",
    "Japanese": "素早い茶色の狐が怠惰な犬を飛び越える。",
    "Arabic": "الثعلب البني السريع يقفز فوق الكلب الكسول.",
    "Code": "def hello():\n    return 'Hello, World!'"
}

print(f"{'Language':<12} {'Chars':<8} {'Tokens':<8} {'Chars/Token':<12}")
print("-" * 44)
for lang, text in texts.items():
    tokens = enc.encode(text)
    ratio = len(text) / len(tokens)
    print(f"{lang:<12} {len(text):<8} {len(tokens):<8} {ratio:<12.1f}")

print()
print("Notice: Non-English text uses more tokens for the same meaning.")
print("Rule of thumb for English: 1 token ~ 4 characters ~ 0.75 words")

In [ ]:
# Cost estimator: how much will this prompt cost?
def estimate_cost(text, model="gpt-4o-mini"):
    """Estimate the cost of input tokens for a given text and model."""
    # Pricing per 1M tokens (as of 2025)
    pricing = {
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "gpt-4o": {"input": 2.50, "output": 10.00},
        "gpt-4-turbo": {"input": 10.00, "output": 30.00},
    }

    enc = tiktoken.encoding_for_model(model)
    token_count = len(enc.encode(text))
    cost_per_token = pricing[model]["input"] / 1_000_000

    return {
        "model": model,
        "tokens": token_count,
        "estimated_input_cost": f"${token_count * cost_per_token:.6f}"
    }

# Compare costs across models for the same text
sample = "Explain the theory of relativity in simple terms." * 20  # ~1000 chars

for model in ["gpt-4o-mini", "gpt-4o", "gpt-4-turbo"]:
    result = estimate_cost(sample, model)
    print(f"{result['model']:<16} {result['tokens']} tokens  cost: {result['estimated_input_cost']}")

---
## 3. LLM API Calls: Generation and Embedding <a id='api-calls'></a>

Two fundamental API call types: generation (produce text) and embedding (produce vectors).

In [ ]:
# Generation: ask the model a question
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is a Transformer in machine learning? Two sentences max."}
    ],
    temperature=0.3,
    max_tokens=100
)

print("Generation response:")
print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.prompt_tokens} input + {response.usage.completion_tokens} output = {response.usage.total_tokens} total")

In [ ]:
# Embedding: convert text to a vector
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Machine learning is a subset of artificial intelligence."
)

embedding = response.data[0].embedding

print(f"Embedding model: text-embedding-3-small")
print(f"Dimensions: {len(embedding)}")
print(f"First 10 values: {[round(v, 4) for v in embedding[:10]]}")
print(f"\nThis vector captures the 'meaning' of the input text.")
print(f"Similar texts will have similar vectors.")

---
## 4. Cosine Similarity and Semantic Search <a id='cosine-similarity'></a>

Cosine similarity measures the angle between two vectors. 
1.0 = identical meaning, 0 = completely unrelated.

In [ ]:
# Calculate cosine similarity between sentence pairs
import numpy as np
from openai import OpenAI

client = OpenAI()

def get_embedding(text):
    response = client.embeddings.create(model="text-embedding-3-small", input=text)
    return response.data[0].embedding

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Test pairs: similar vs different
pairs = [
    ("I love dogs", "I adore puppies"),
    ("I love dogs", "I hate dogs"),
    ("I love dogs", "The stock market crashed"),
    ("Python is a programming language", "Python is used for coding"),
    ("Python is a programming language", "A python is a large snake"),
]

print(f"{'Text A':<35} {'Text B':<35} {'Similarity':<10}")
print("-" * 82)

for text_a, text_b in pairs:
    emb_a = get_embedding(text_a)
    emb_b = get_embedding(text_b)
    sim = cosine_similarity(emb_a, emb_b)
    print(f"{text_a:<35} {text_b:<35} {sim:.3f}")

In [ ]:
# Mini semantic search engine
documents = [
    "Python is a versatile programming language used for web development and AI.",
    "Machine learning algorithms learn patterns from data to make predictions.",
    "Neural networks are inspired by the human brain's structure.",
    "APIs allow different software applications to communicate with each other.",
    "Docker containers package applications with their dependencies.",
    "Cloud computing provides on-demand access to computing resources.",
    "Natural language processing helps computers understand human language.",
    "Data visualization helps communicate insights from complex datasets."
]

# Embed all documents (batch for efficiency)
response = client.embeddings.create(model="text-embedding-3-small", input=documents)
doc_embeddings = [d.embedding for d in response.data]

def search(query, top_k=3):
    """Search documents by semantic similarity."""
    query_emb = get_embedding(query)
    scores = [(i, cosine_similarity(query_emb, doc_emb)) for i, doc_emb in enumerate(doc_embeddings)]
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# Try different queries
queries = [
    "How do machines learn?",
    "What is containerization?",
    "How can I build an AI chatbot?"
]

for query in queries:
    print(f"Query: '{query}'")
    for idx, score in search(query):
        print(f"  {score:.3f}: {documents[idx][:70]}...")
    print()

---
## 5. Streaming Inference <a id='streaming'></a>

Tokens are generated one at a time. Streaming shows them immediately instead of waiting for the full response.

In [ ]:
# Streaming demo: watch tokens arrive one at a time
import time
from openai import OpenAI

client = OpenAI()

print("Streaming response (tokens arrive in real-time):\n")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a 4-line poem about coding."}],
    stream=True,
    temperature=0.7
)

token_count = 0
start = time.time()

for chunk in stream:
    if chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        print(content, end="", flush=True)
        token_count += 1

elapsed = time.time() - start
print(f"\n\n--- Stats ---")
print(f"Tokens generated: ~{token_count}")
print(f"Time: {elapsed:.2f}s")
print(f"Speed: ~{token_count/elapsed:.0f} tokens/second")

---
## 6. Temperature Comparison <a id='temperature'></a>

Temperature controls randomness. Let's see the same prompt at different temperatures.

In [ ]:
# Same prompt, different temperatures
from openai import OpenAI

client = OpenAI()

prompt = "Complete this sentence creatively: The robot looked at the sunset and"
temperatures = [0, 0.3, 0.7, 1.0, 1.5]

print(f"Prompt: '{prompt}'\n")
print(f"{'Temp':<6} Response")
print("-" * 80)

for temp in temperatures:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=60
    )
    text = response.choices[0].message.content.strip()
    # Truncate for display
    display = text[:90] + "..." if len(text) > 90 else text
    print(f"{temp:<6} {display}")

In [ ]:
# Visualize: run the same prompt 5 times at each temperature to see variance
import matplotlib.pyplot as plt
from openai import OpenAI

client = OpenAI()

prompt = "Name one color."
temps = [0, 0.5, 1.0, 1.5]
runs_per_temp = 5

results = {}
for temp in temps:
    responses = []
    for _ in range(runs_per_temp):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=10
        )
        responses.append(r.choices[0].message.content.strip().lower())
    results[temp] = responses
    unique = len(set(responses))
    print(f"Temp {temp}: {responses}  (unique: {unique}/{runs_per_temp})")

# Plot unique responses per temperature
fig, ax = plt.subplots(figsize=(8, 4))
unique_counts = [len(set(results[t])) for t in temps]
ax.bar([str(t) for t in temps], unique_counts, color=["#2196F3", "#4CAF50", "#FF9800", "#F44336"])
ax.set_xlabel("Temperature")
ax.set_ylabel("Unique Responses (out of 5)")
ax.set_title("Temperature vs Response Diversity")
ax.set_ylim(0, runs_per_temp + 0.5)
plt.tight_layout()
plt.show()

print("\nLow temp = consistent. High temp = diverse (and sometimes wrong).")

---
## 7. Attention Heatmap Visualization <a id='attention'></a>

This is a **simulated** attention heatmap to build intuition about how self-attention works.
In a real Transformer, every token computes attention scores with every other token.

In [ ]:
# Simulated attention heatmap
import numpy as np
import matplotlib.pyplot as plt

# Simulate attention for the sentence "The cat sat on the mat because it was tired"
tokens = ["The", "cat", "sat", "on", "the", "mat", "because", "it", "was", "tired"]
n = len(tokens)

# Create a plausible attention pattern:
# "it" (index 7) should attend strongly to "cat" (index 1)
# Adjacent words attend to each other
# Diagonal is strong (self-attention)
np.random.seed(42)
attention = np.random.uniform(0.02, 0.08, (n, n))

# Make diagonal strong (each token attends to itself)
for i in range(n):
    attention[i, i] = np.random.uniform(0.3, 0.5)

# "it" attends strongly to "cat" (pronoun resolution)
attention[7, 1] = 0.45

# "tired" attends to "cat" (subject of the clause)
attention[9, 1] = 0.30

# "sat" attends to "cat" (subject-verb)
attention[2, 1] = 0.35

# "mat" attends to "sat" and "on"
attention[5, 2] = 0.25
attention[5, 3] = 0.28

# Normalize rows to sum to 1 (like softmax)
attention = attention / attention.sum(axis=1, keepdims=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(attention, cmap="Blues", aspect="auto")

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=11)
ax.set_yticklabels(tokens, fontsize=11)
ax.set_xlabel("Attending TO (Key)", fontsize=12)
ax.set_ylabel("Attending FROM (Query)", fontsize=12)
ax.set_title('Simulated Self-Attention: "The cat sat on the mat because it was tired"', fontsize=13)

# Add values in cells
for i in range(n):
    for j in range(n):
        color = "white" if attention[i, j] > 0.25 else "black"
        ax.text(j, i, f"{attention[i, j]:.2f}", ha="center", va="center", fontsize=8, color=color)

plt.colorbar(im, label="Attention Weight")
plt.tight_layout()
plt.show()

print('Notice: "it" (row 7) attends strongly to "cat" (col 1).')
print("This is how Transformers resolve pronoun references across distance.")

---
## 8. Exercises <a id='exercises'></a>

### Exercise 1: Token Cost Calculator

Write a function that takes a document (string) and prints:
- Token count for gpt-4o-mini, gpt-4o, and gpt-4-turbo
- Estimated input cost for each model
- Which model is most cost-effective

Hint: use `tiktoken.encoding_for_model()` for each model.

In [ ]:
# Exercise 1: Your code here

def document_cost_report(document: str):
    """Print a cost comparison across models for the given document."""
    # TODO: implement this
    pass

# Test with a sample document
sample_doc = """
Artificial intelligence has transformed how we build software.
Large language models can understand and generate human language
with remarkable fluency. AI Engineers build applications that
leverage these models through APIs, retrieval pipelines, and
agent frameworks.
""" * 10  # Repeat to simulate a longer document

# document_cost_report(sample_doc)

### Exercise 2: Semantic Search Over a Custom Knowledge Base

Build a search system over at least 10 documents of your choice (could be FAQ entries,
Wikipedia paragraphs, or any text). Implement search and print the top 3 results with scores.

In [ ]:
# Exercise 2: Your code here

# Step 1: Define your knowledge base (at least 10 documents)
knowledge_base = [
    # TODO: add your documents
]

# Step 2: Embed all documents

# Step 3: Implement search function

# Step 4: Test with 3 different queries

### Exercise 3: Find the Best Temperature for Classification

Use gpt-4o-mini to classify movie reviews as "positive" or "negative".
Run each review at temperatures 0, 0.5, and 1.0 five times each.
Which temperature gives the most consistent (correct) results?

In [ ]:
# Exercise 3: Your code here

reviews = [
    ("This movie was absolutely fantastic! Best film of the year.", "positive"),
    ("Terrible waste of time. The plot made no sense at all.", "negative"),
    ("A masterpiece of storytelling with incredible performances.", "positive"),
    ("I fell asleep halfway through. So boring.", "negative"),
    ("Decent movie, nothing special but enjoyable enough.", "positive"),
]

# TODO: For each review, at each temperature, run 5 times
# Count how many times the model gets the right label
# Print accuracy per temperature

---
## Checkpoint

Before moving on to Session 2, verify:

- [ ] You can tokenize text and estimate API costs
- [ ] You understand the difference between generation and embedding API calls
- [ ] You can compute cosine similarity and build basic semantic search
- [ ] You've seen streaming in action and understand why it exists
- [ ] You understand how temperature affects output randomness

### Next: Session 2 - Building with LLMs

We'll use these foundations to build real applications: prompt engineering, chunking, retrieval, and RAG.